# Среднее число жителей в заселённых квартирах

PPF — People Per Flat. Характеризует только те квартиры, в которых кто-то постоянно живёт. Соответственно, если в квартире никто постоянно не живёт, то для неё этот показатель неприменим. Благодаря этой особенности можно использовать PPF для оценки вакантности. Как именно — см. тетрадку №4.

PPF считаем по данным [переписи 2021 года в Москве](https://77.rosstat.gov.ru/folder/210976). В томе 11 есть таблица 6: «Жилые помещения по их типам, числу комнат и числу проживающих в них частных домохозяйств по муниципальным образованиям». Из неё можно понять, сколько человек в среднем проживает в одной квартире с учётом числа комнат по муниципалитетам. Расчёт получается не самый тривиальный, так как данные идут в разбивке по (1) числу домохозяйств в одной квартире и (2) числу проживающих в домохозяйстве, но если всё аккуратно перемножить и сложить, а потом поделить (формула есть в коде), то получим среднее число постоянно проживающих в одной заселённой квартире в разбивке по муниципалитетам и числу комнат в квартире (1–4, `-1` значит среднее независимо от числа комнат). Разбивка по числу комнат важна, чтобы получить более точный результат — в тетрадке №3 оцениваем число квартир с разным числом комнат по домам.

Так как данные переписи собираются bottom-up, то есть от конкретных домохозяйств, то пустующие квартиры в указанной выше таблице не учитываются. Таким образом, мы, грубо говоря, получаем косвенный результат поквартирного обхода и на его основе пытаемся потом определить исходные данные по вакантности. Из методики переписи мы также получаем некоторое определение вакантности: это жилое помещение, в котором постоянно никто не живёт. Подробнее см. [методологические пояснения к переписи](https://rosstat.gov.ru/storage/mediabank/%D0%9E%D0%B1%D1%89%D0%B0%D1%8F%20%D1%87%D0%B0%D1%81%D1%82%D1%8C_Met_VPN-2020.docx), с. 3 с «Население переписано по месту своего постоянного (обычного) жительства».

In [17]:
from fuzzywuzzy import process
import pandas as pd

`liv_cond` — результаты переписи, `repair_data` — набор открытых данных ФРТ «[Многоквартирные дома в региональной программе капитального ремонта по городу Москве (отчет КР 1.1)](https://xn--80adsazqn.xn--p1aee.xn--p1ai/opendata/export/184)». Используем именно их, а не те же, что в геокодировании, так как здесь есть разбивка по муниципальным образованиям. Набор нужен, чтобы привести наименования муниципалитетов к тем, что используются в открытых данных ФРТ, с которыми мы дальше работаем.

In [18]:
liv_cond = pd.read_excel("data/census_results_11_6.xlsx", sheet_name=None, na_values=["-"])
repair_data = pd.read_csv("data/export-kr1_1-77-20250301.zip", sep=";")

In [24]:
rows = []
for sheet_name, data in liv_cond.items():   
    part = data.iloc[[17, 19, 20, 21, 22]].copy()
    part.columns = [f"c{i}" for i in range(part.shape[1])]
    part["ppf"] = (
        (
            part["c3"] + part["c4"] * 2 + part["c5"] * 3 + part["c6"] * 4 + part["c8"]
             + part["c10"] + part["c12"] + part["c14"]
        ) / part["c1"]
    )        
    part["sheet_name"] = sheet_name
    part["n_rooms"] = [-1] + list(range(1, 5))    
    rows.append(part[["sheet_name", "ppf", "n_rooms"]])

ppf_by_sheet = pd.concat(rows)
ppf_by_sheet.head()

,sheet_name,ppf,n_rooms
17,г. Москва,2.880098,-1
19,г. Москва,1.950212,1
20,г. Москва,2.760971,2
21,г. Москва,3.690136,3
22,г. Москва,4.669829,4


In [26]:
mo = repair_data["mun_obr"].unique()
sheet_names = ppf_by_sheet["sheet_name"].unique()

In [45]:
MO_MAPPING = {
    "Савёловский район": "Савеловское",
    "Нагорный район": "Нагорное",
    "район Северный": "Северное",
    "район Восточный": "Восточное",
    "район Хорошёво-Мнёвники": "Хорошево-Мневники",
    "Бабушкинский район": "Бабушкинское",
    "городской округ Троицк": "Поселение Троицк",
    "городской округ Щербинка": "Поселение Щербинка",
    "Хорошёвский район": "Хорошевское",
    "Пресненский район": "Пресненское",
    "Рязанский район": "Рязанское",
    "район Измайлово": "Измайлово",
    "Коммунарка": 'Поселение "Мосрентген"',
} # manual fixes of wrong fuzzy search results

parts = []
for mun_obr in sorted(mo):
    sheet_name = MO_MAPPING.get(mun_obr)
    if sheet_name is None:
        sheet_name = process.extractOne(mun_obr, sheet_names, score_cutoff=70)
        if sheet_name is None:
            print(f"Cannot find matching municipality for {mun_obr}")
            continue
        else:
            sheet_name = sheet_name[0]
    print(mun_obr, "~", sheet_name)
    
    part = ppf_by_sheet.loc[ppf_by_sheet["sheet_name"] == sheet_name].copy()     
    part["mun_obr"] = mun_obr
    part["ppf"] = part["ppf"].astype(float)
    parts.append(part)

ppf_by_mo = pd.concat(parts)

Академический район ~ Академическое
Алексеевский район ~ Алексеевское
Алтуфьевский район ~ Алтуфьевское
Арбат ~ Арбат
Бабушкинский район ~ Бабушкинское
Басманный район ~ Басманное
Бескудниковский район ~ Бескудниковское
Богородское ~ Богородское
Бутырский район ~ Бутырское
Внуково ~ Внуково
Войковский ~ Войковское
Войковский район ~ Войковское
Восточный ~ Восточное
Гагаринский ~ Гагаринское
Гагаринский район ~ Нагатинский Затон
Головинский ~ Головинское
Головинский район ~ Головинское
Даниловский ~ Даниловское
Даниловский район ~ Даниловское
Дмитровский район ~ Дмитровское
Донской район ~ Донское
Западное Дегунино ~ Западное Дегунино
Ивановское ~ Ивановское
Коммунарка ~ Поселение "Мосрентген"
Косино-Ухтомский ~ Косино-Ухтомское
Красносельский район ~ Красносельское
Кунцево ~ Кунцево
Левобережный ~ Левобережное
Лефортово ~ Лефортово
Ломоносовский район ~ Ломоносовское
Лосиноостровский ~ Лосиноостровское
Лосиноостровский район ~ Лосиноостровское
Люблино ~ Люблино
Марфино ~ Марфино
Марьин

Листы, соответствующие более крупным территориальным единицам, ожидаемо ничему не соответствуют. В `repair_data` есть варианты именования одного и того же муниципального образования (например, «Арбат» и «Район Арбат»), поэтому в итоговой таблице `mun_obr` соответствует `repair_data["mun_obr"]`. Логирование позволяет вручную проверить правильность сопоставления муниципалитетов, так как автоматический поиск через `fuzzywuzzy` не всегда даёт корректный результат (исключения помещаются в `MO_MAPPING`).

In [37]:
ppf_by_mo.head(10)

,sheet_name,ppf,n_rooms,mun_obr
17,Академическое,3.113928,-1,Академический район
19,Академическое,2.138218,1,Академический район
20,Академическое,2.862416,2,Академический район
21,Академическое,3.583501,3,Академический район
22,Академическое,4.734848,4,Академический район
17,Алексеевское,2.828805,-1,Алексеевский район
19,Алексеевское,2.105233,1,Алексеевский район
20,Алексеевское,2.818257,2,Алексеевский район
21,Алексеевское,3.820906,3,Алексеевский район
22,Алексеевское,4.963746,4,Алексеевский район


In [46]:
ppf_by_mo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 980 entries, 17 to 22
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   sheet_name  980 non-null    object 
 1   ppf         944 non-null    float64
 2   n_rooms     980 non-null    int64  
 3   mun_obr     980 non-null    object 
dtypes: float64(1), int64(1), object(2)
memory usage: 38.3+ KB


Пропуски возникли из-за того, что по некоторым муниципальных образованиях не было каких-то данных переписи. Ничего с ними не делаем, так как их мало. 

Проверим, сильно ли отличается значение `ppf` в разных муниципалитетах.

In [47]:
ppf_by_mo.groupby("n_rooms")["ppf"].describe()

,count,mean,std,min,25%,50%,75%,max
n_rooms,,,,,,,,
-1,196.0,2.863233,0.337446,1.993626,2.650477,2.875841,3.050019,3.810079
1,179.0,1.974923,0.288657,1.331245,1.764249,1.964223,2.151637,2.751795
2,190.0,2.782573,0.286980,1.951057,2.605281,2.778228,2.956248,3.627553
3,189.0,3.694761,0.316041,2.690863,3.539228,3.712145,3.918247,4.743994
4,190.0,4.672277,0.541706,2.480978,4.405981,4.737073,5.118507,5.664198


Видно, что некоторая изменчивость есть, но не очень большая: межквартильный размах и стандартное отклонение составляют где-то 10–15% от среднего значения в каждой группе по числу комнат.

Сохраняем результат для дальнейшего использования.

In [48]:
ppf_by_mo.to_parquet("data/ppf_by_mo.parquet", index=False)